# Groceries — Market Basket Analysis

EDA y reglas de asociación (soporte / confianza / lift) para **co-compra**.

- Datos: `data/kaggle/groceries-mba/Groceries_dataset.csv`
- Uso Culebra: analogía **lista de 8** e impulso (miel+loncheado, cesta+tote)
- Mapa: `docs/Variables_Decision_Datasets_Kaggle.md`


In [ ]:
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.figsize"] = (11, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25


## Cargar dataset

In [ ]:
here = Path.cwd()
candidates = [
    here / "Groceries_dataset.csv",
    here / "data" / "kaggle" / "groceries-mba" / "Groceries_dataset.csv",
    Path("/kaggle/input"),
]
path = None
for c in candidates:
    if c.is_file():
        path = c
        break
    if c.is_dir():
        found = list(c.rglob("Groceries_dataset.csv"))
        if found:
            path = found[0]
            break
if path is None:
    raise FileNotFoundError("No se encuentra Groceries_dataset.csv")

raw = pd.read_csv(path)
raw["Date"] = pd.to_datetime(raw["Date"], dayfirst=True, errors="coerce")
raw["itemDescription"] = raw["itemDescription"].astype(str).str.strip()
print("Dataset path:", path.resolve())
print(raw.shape)
raw.head()


## Resumen rápido

In [ ]:
# Una cesta = Member_number + Date (misma compra)
raw["basket_id"] = raw["Member_number"].astype(str) + "_" + raw["Date"].dt.strftime("%Y-%m-%d")
baskets = (
    raw.groupby("basket_id")["itemDescription"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="items")
)
baskets["n_items"] = baskets["items"].str.len()

print("Cestas:", len(baskets))
print("Ítems únicos:", raw["itemDescription"].nunique())
print("Media ítems/cesta:", round(baskets["n_items"].mean(), 2))
print("% cestas multi-ítem (≥2):", round((baskets["n_items"] >= 2).mean() * 100, 1), "%")
display(baskets["n_items"].describe().to_frame("n_items"))


## Top productos (proxy lista de 8)

In [ ]:
top_n = 15
item_counts = raw["itemDescription"].value_counts().head(top_n)

fig, ax = plt.subplots()
item_counts.sort_values().plot(kind="barh", ax=ax, color="#2f5d50")
ax.set_title(f"Top {top_n} productos (frecuencia en líneas)")
ax.set_xlabel("Líneas de ticket")
plt.tight_layout()
plt.show()

# Analogía grosera Culebra (no son los mismos SKU; sirve para practicar lift)
culebra_analog = {
    "whole milk": "queso / lácteo",
    "other vegetables": "picos / snack",
    "rolls/buns": "picos",
    "soda": "vino / bebida",
    "yogurt": "miel / mermelada",
    "bottled water": "tote / impulso barra",
    "root vegetables": "loncheado",
    "tropical fruit": "mini-cata / experiencia",
}
print("Analogía orientativa Culebra (solo para interpretar pares):")
for k, v in culebra_analog.items():
    if k in item_counts.index:
        print(f"  {k:20s} → {v}")


## Tamaño de cesta — proxy de impulso / attach

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
baskets["n_items"].clip(upper=12).hist(bins=12, ax=axes[0], color="#2f5d50", edgecolor="white")
axes[0].set_title("Distribución ítems por cesta (cap 12)")
axes[0].set_xlabel("n_items")

daily = raw.groupby(raw["Date"].dt.to_period("M")).size()
daily.index = daily.index.to_timestamp()
daily.plot(ax=axes[1], color="#8b4513")
axes[1].set_title("Líneas de ticket por mes")
axes[1].set_ylabel("líneas")
plt.tight_layout()
plt.show()

attach_pct = (baskets["n_items"] >= 2).mean() * 100
print(f"Attach proxy (≥2 ítems): {attach_pct:.1f}%  · meta Culebra impulso ≥ 40%")


## Pares con mayor lift (qué reforzar juntos en caja)

In [ ]:
MIN_SUPPORT = 0.01  # 1% de cestas
TOP_ITEMS = 40      # limitar combinatoria

freq = raw["itemDescription"].value_counts()
keep = set(freq.head(TOP_ITEMS).index)
n_baskets = len(baskets)

item_support = {
    item: (baskets["items"].apply(lambda xs: item in xs).mean())
    for item in keep
}

pair_counts = {}
for items in baskets["items"]:
    filtered = [i for i in items if i in keep]
    for a, b in combinations(sorted(filtered), 2):
        pair_counts[(a, b)] = pair_counts.get((a, b), 0) + 1

rows = []
for (a, b), cnt in pair_counts.items():
    support = cnt / n_baskets
    if support < MIN_SUPPORT:
        continue
    conf_ab = support / item_support[a] if item_support[a] else 0
    conf_ba = support / item_support[b] if item_support[b] else 0
    lift = support / (item_support[a] * item_support[b]) if item_support[a] * item_support[b] else 0
    rows.append(
        {
            "antecedent": a,
            "consequent": b,
            "support": support,
            "confidence_a→b": conf_ab,
            "confidence_b→a": conf_ba,
            "lift": lift,
            "count": cnt,
        }
    )

rules = pd.DataFrame(rows).sort_values("lift", ascending=False)
print(f"Pares con support ≥ {MIN_SUPPORT:.0%} (top {TOP_ITEMS} ítems): {len(rules)}")
display(rules.head(20).round(3))

top_plot = rules.head(12).copy()
top_plot["pair"] = top_plot["antecedent"] + " + " + top_plot["consequent"]
fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(top_plot["pair"][::-1], top_plot["lift"][::-1], color="#2f5d50")
ax.axvline(1.0, color="crimson", ls="--", label="lift = 1 (independientes)")
ax.set_xlabel("lift")
ax.set_title("Top pares por lift — candidatos a colocar juntos")
ax.legend()
plt.tight_layout()
plt.show()


## Modelo: ¿cesta multi-ítem? (clasificación attach)

In [ ]:
# Features: presencia de top ítems (one-hot ligero) → predecir n_items ≥ 2
top_feat = list(freq.head(20).index)
X = pd.DataFrame({item: baskets["items"].apply(lambda xs: int(item in xs)) for item in top_feat})
# Quitar leakage: no usar el conteo directo; sí presencia de anclas
y = (baskets["n_items"] >= 2).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
print("Accuracy multi-ítem:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred, digits=3))

imp = pd.Series(clf.feature_importances_, index=top_feat).sort_values(ascending=False).head(12)
fig, ax = plt.subplots()
imp.sort_values().plot(kind="barh", ax=ax, color="#8b4513")
ax.set_title("Importancia: productos que predicen cesta multi-ítem")
plt.tight_layout()
plt.show()


## Conclusiones para decidir (Culebra)

1. **Attach**: medir `% tickets con ≥2 líneas` en showroom (meta impulso ≥ 40%).
2. **Pares con lift > 1**: candidatos a colocación conjunta en isla / caja (traducir a miel+loncheado, cesta+tote, vino+picos).
3. **Anclas**: productos cuya presencia predice multi-ítem → priorizar stock y visibilidad en la lista de 8.
4. Este dataset **no tiene precios €**; calibrar con CSV quincenal / sintético Culebra.
